# GAN Coloring

![](https://live.staticflickr.com/65535/54531085276_a4c5d63f4f_b.jpg)

*The images were generated using the Flux-dev model and then edited by the task author.*

# Introduction

The topic of this task is image enhancement, which is a collective term covering such operations as:

* image super-resolution,
* image denoising,
* image deblurring,
* low-light image enhancement,
* **image colorization**,
* and many others.

The goal of these techniques is to obtain a high-quality image from a low-quality input image. This process often requires filling in missing information based on context, which is why deep learning methods are commonly used.

Thanks to such techniques, we can in a sense “look” into the past from a new perspective. An example is the image of Warsaw from the early interwar period — on the left in its original form, and on the right in a version colorized by Mariusz Zając.

![](https://live.staticflickr.com/65535/54531426695_24b5710613_b.jpg)

In this task, we focus on colorizing black-and-white photographs of human faces.

### Image Colorization

A color image is described by three RGB channels (red, green, blue). It can be easily transformed into a monochrome image described by a single channel. However, when converting to grayscale, a simple arithmetic mean of RGB values is not used, because it does not correspond to the way the human eye perceives brightness of different colors (e.g., we perceive blue light as darker than green light).
Therefore, pixels in a grayscale image are computed using the formula:

$$ x^{(gray)} = 0.299 \cdot x^{(red)} + 0.587 \cdot x^{(green)} + 0.114 \cdot x^{(blue)}. $$

It is worth noting that conversion from RGB to grayscale is deterministic and easy to perform. The inverse operation — recovering color information — is not deterministic, because many different combinations of RGB values can result in the same grayscale value. This means that image colorization is an ill-posed problem, implying that for a single grayscale image many correct color versions can be generated.

For example, a black-and-white image of a car is difficult to colorize unambiguously — the vehicle could have been almost any color. On the other hand, models trained on appropriate data are able to exploit statistical regularities — such as grass usually being green, the sky blue, and tigers orange and black — to generate the most probable color versions.

Formally, the colorization task is defined as learning a model $\mathcal{M}_\theta$ with parameters $\theta$, which takes a grayscale image $x^{(gray)}$ as input and outputs its color version $\hat{x}^{(rgb)}$:

$$ \hat{x}^{(rgb)} = \mathcal{M}_\theta(x^{(gray)}). $$

We would like the model predictions $\hat{x}^{(rgb)}$ to closely resemble the original color images $x^{(rgb)}$. To achieve this, we aim to find such parameter values $\theta$ that minimize a chosen difference measure (e.g., mean squared error) between predictions and ground truth images:

$$ \theta^* = \arg \min_{\theta} \sum_{i = 1}^N d(x^{(rgb)}*i, \mathcal{M}*\theta(x^{(gray)}_i)), $$

where $N$ denotes the number of samples in the training set.

### Generative Adversarial Networks

Generative adversarial networks (GANs) were one of the first approaches addressing the problem of data generation using deep learning techniques. Since the generation process does not require any input to the network, it is not straightforward to define a loss function, because we do not know what output to expect from the network. For example, a network may generate a realistic face image, but when compared to another random face image, the loss would be very large despite the output being correct.

For this reason, the GAN architecture consists of two networks: a generator and a discriminator.

* The role of the generator is to generate new random images.
* The role of the discriminator is to predict whether the input image comes from the dataset or was generated by the generator.

The generator tries to fool the discriminator (maximizes the discriminator’s classification error), while the discriminator tries to correctly determine whether the image is real or fake (minimizes its classification error). Both networks are trained alternately until convergence.

In this task, we will not train a GAN; instead, we will use a previously trained generator model that can produce random face images of size $256 \times 256$ from noise. The architecture of this network is called StyleGAN and is shown in the diagram below. It is divided into two modules: the mapping network $f$ and the synthesis network $g$.

The network $f$ is an MLP that transforms a random vector $z$, containing $512$ elements sampled from a normal distribution $\mathcal{N}(0, \mathbb{I})$, into a $512$-dimensional vector $w$, which is used to condition the network $g$. We can interpret $w$ as a latent representation of the generated image.

The network $g$ is a convolutional network with a fixed $4 \times 4$ input, which progressively increases its resolution up to $256 \times 256$. In each block, the network $g$ is conditioned by the vector $w$, providing diversity in the generated images. Additionally, some noise is also fed into the network, further increasing variability.

![](https://live.staticflickr.com/65535/54531085226_a4fe9a0c2e_z.jpg)


# Task

In this task, given a pretrained `generator` capable of producing $256 \times 256$ face images, you are required to propose a method for colorizing black-and-white face images of the same resolution. The goal is to extract the knowledge embedded in the generator’s weights and reuse it for a completely different task, previously unknown to the generator.

### Data

In this task, there is no training dataset available. Only validation data is provided for an initial evaluation of your proposed approach. The validation set contains 500 paired grayscale images with their corresponding color versions.

### Evaluation criterion

To evaluate the quality of your solution, a metric composed of two sub-metrics will be used: *PSNR* and *LPIPS*.

PSNR is inversely proportional to the mean squared error between the generated sample $\hat{x}^{(rgb)}$ and the ground-truth color image $x^{(rgb)}$. It is defined as:
$$ PSNR = 10 \cdot \log_{10}\left(\frac{\text{range}^2}{MSE(\hat{x}^{(rgb)}, x^{(rgb)})}\right), $$
where $\text{range}$ is the range of possible values that the images $\hat{x}^{(rgb)}$ and $x^{(rgb)}$ can take. PSNR is to be maximized.
The scored range for this metric is $(22.0, 26.0)$ — points are assigned proportionally.

LPIPS is the average $L1$ distance between feature maps extracted from selected layers of an `AlexNet` network, computed from inputs $\hat{x}^{(rgb)}$ and $x^{(rgb)}$. LPIPS is to be minimized.
The scored range for this metric is $(0.15, 0.11)$ — points are assigned proportionally.

The final score is the weighted average of the scores from PSNR and LPIPS, with weights $0.25$ (PSNR) and $0.75$ (LPIPS).

### Constraints

* Your solution will be tested on the Competition Platform without internet access and in a GPU-enabled environment.
* The evaluation of your final solution on the Competition Platform must not exceed 5 minutes on GPU.
* Allowed libraries: torch, numpy, torchvision, pillow.

### Submission files

You should submit only this notebook with your solution filled in (see `YourModel` class).

### Evaluation

During grading, the flag `FINAL_EVALUATION_MODE` will be set to `True`.

You can score between 0 and 100 points for this task. The score will be computed (on a hidden test set) on the Competition Platform using the formula above and rounded to an integer. If your solution does not meet the requirements or fails to run properly, you will receive 0 points.


# Starter code


In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

FINAL_EVALUATION_MODE = False

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

import os
import torch
import tarfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torchvision.transforms as T

from PIL import Image
from io import BytesIO
from torch.utils.data import Dataset, DataLoader
from torchmetrics.image import PeakSignalNoiseRatio as PSNR
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity as LPIPS


from stylegan import load_generator
RANDOM_SEED = 1

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

The cell below contains evaluation and visualization functions for your solutions, as well as the validation dataset.


In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

import math

def round_half_up(number: float) -> int:
    return int(math.floor(number + 0.5))

def plot_batch(batch):
    """ Function for displaying a batch of images (generated or real). Displays up to 8 images """
    to_show = min(len(batch), 8)

    batch = batch.to('cpu')
    batch = batch * 0.5 + 0.5
    batch = batch.clamp(0, 1)
    batch = batch.permute(0, 2, 3, 1).numpy()

    fig, axes = plt.subplots(1, to_show, figsize=(to_show * 3, 3))

    if to_show == 1:
        axes = [axes]

    for ax, img in zip(axes, batch[:to_show]):
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


class TarImageDataset(Dataset):
    """ Dataset class used for validating your method """
    def __init__(self, tar_path):
        self.tar_path = tar_path
        self.tar = tarfile.open(tar_path, 'r')
        self.gt_paths = sorted([m.name for m in self.tar.getmembers() if m.name.startswith('data/GT/') and m.name.endswith('.jpeg')])
        self.gray_paths = [p.replace('GT', 'GRAY') for p in self.gt_paths]

        self.transform = T.Compose([T.ToTensor(), T.Normalize(mean=[0.5], std=[0.5])])

    def __len__(self):
        return len(self.gt_paths)

    def __getitem__(self, idx):
        """ returns a grayscale image and a color image """
        gt_member = self.tar.getmember(self.gt_paths[idx])
        gray_member = self.tar.getmember(self.gray_paths[idx])

        gt_image = Image.open(BytesIO(self.tar.extractfile(gt_member).read())).convert('RGB')
        gray_image = Image.open(BytesIO(self.tar.extractfile(gray_member).read())).convert('L')

        return (
            self.transform(gray_image),
            self.transform(gt_image)
        )

    def __del__(self):
        """ Closes the file when an object of this class is deleted """
        if hasattr(self, 'tar') and self.tar:
            self.tar.close()


def validate_solution(your_model, device, split="val"):
    dataset = TarImageDataset(f"./data/{split}.data")
    dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

    lpips_metric = LPIPS(net_type='alex').to(device)
    psnr_metric = PSNR(data_range=2.0).to(device)

    your_model.eval()

    with torch.no_grad():
        for x_gray, x_gt in dataloader:
            x_gray = x_gray.to(device)
            x_gt = x_gt.to(device)

            x_pred = your_model.predict(x_gray).clamp_(-1, 1)

            psnr_metric.update(x_pred, x_gt)
            lpips_metric.update(x_pred, x_gt)

    avg_psnr = psnr_metric.compute()
    avg_lpips = lpips_metric.compute()

    print("PSNR: ", avg_psnr)
    print("LPIPS:", avg_lpips)

    PSNR_MIN, PSNR_MAX = 22, 26
    LPIPS_MIN, LPIPS_MAX = 0.11, 0.15

    psnr_points = ((torch.clamp(avg_psnr, PSNR_MIN, PSNR_MAX) - PSNR_MIN) / (PSNR_MAX - PSNR_MIN)).item()
    lpips_points = ((LPIPS_MAX - torch.clamp(avg_lpips, LPIPS_MIN, LPIPS_MAX)) / (LPIPS_MAX - LPIPS_MIN)).item()
    total_points = psnr_points * 0.25 + lpips_points * 0.75

    psnr_points = round_half_up(psnr_points * 100)
    lpips_points = round_half_up(lpips_points * 100)
    total_points = round_half_up(total_points * 100)

    return psnr_points, lpips_points, total_points


def show_image_grid(inputs, predictions, targets):
    """
    function that visualizes inputs, predictions, and target images
    """
    def tensor_to_numpy(img):
        img = (img.clamp(-1, 1) + 1) / 2
        img = img.cpu().numpy()
        if img.shape[0] == 1:
            return img.squeeze(0)
        return np.transpose(img, (1, 2, 0))

    titles = ["Inputs", "Model predictions", "Target images"]
    images = [inputs, predictions, targets]

    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    for row in range(3):
        for col in range(4):
            img = tensor_to_numpy(images[row][col])
            cmap = 'gray' if images[row].shape[1] == 1 else None
            axes[row, col].imshow(img, cmap=cmap)
            axes[row, col].axis('off')
        axes[row, 0].text(-0.2, 0.5, titles[row], va='center', ha='right',
                          fontsize=14, transform=axes[row, 0].transAxes)

    plt.suptitle("Example images from the validation set")
    plt.tight_layout()
    plt.show()


def show_score_bars(psnr_score, lpips_score, task_score):
    """
    Function that visualizes obtained scores as bar charts
    """
    labels = ["PSNR score", "LPIPS score", "task score"][::-1]
    values = [psnr_score, lpips_score, task_score][::-1]

    plt.style.use('ggplot')

    fig, ax = plt.subplots(figsize=(16, 4))
    y = np.arange(len(labels))
    norm = mcolors.Normalize(vmin=0, vmax=100)
    cmap = plt.get_cmap('RdYlGn')
    colors = [cmap(norm(v)) for v in values]

    ax.barh(y, values, color=colors)
    ax.set_xlim(0, 100)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Scores")
    ax.set_title("Model evaluation")
    for i, v in enumerate(values):
        ax.text(v + 1, i, f"{v:.1f}", va='center', fontsize=10)

    plt.tight_layout()
    plt.show()

    # Reset to default style
    plt.style.use('default')


def benchmark_solution(your_model, device, split="val"):
    """
    Function that validates the model on validation data and visualizes the obtained results
    """
    dataset = TarImageDataset(f"./data/{split}.data")
    dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

    x_gray, x_gt = next(iter(dataloader)) 
    x_pred = your_model.predict(x_gray.to(device)).cpu().clamp_(-1, 1)

    psnr_points, lpips_points, total_points = validate_solution(your_model, device, split)
    
    show_image_grid(x_gray, x_pred, x_gt)
    show_score_bars(psnr_points, lpips_points, total_points)



In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

latent_dim = 512   # size of the StyleGAN latent representation
size = 256         # resolution of images generated by the StyleGAN model
device = 'cuda' if torch.cuda.is_available() else 'cpu'

if device == 'cpu':
    print('Warning: no GPU available on the machine!')

# Loading the StyleGAN generator.
# Its implementation is located in the stylegan.py file.
# A detailed analysis of the model code is not prohibited, but is not recommended
generator = load_generator()
generator.to(device)

print('Model loaded successfully')

The cell below contains code for generating faces using the StyleGAN model. Generation is demonstrated in two versions. The first version is more detailed and consists of the following steps:

* first, we sample $z$ from a normal distribution
* then, using the `get_latent` method, we apply the model $f(z)$ to obtain the $w$ vectors
* finally, we call the `style_to_image` method, which uses the $g$ network to generate face images conditioned on the $w$ vector

Alternatively, images can be generated using the `forward` method, which combines steps 2 and 3.


In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

if not FINAL_EVALUATION_MODE:
    with torch.no_grad():
        z = torch.randn(8, latent_dim, device=device) 
        w = generator.get_latent(z)
        sample = generator.style_to_image(w)
        plot_batch(sample)

        z = torch.randn(8, latent_dim, device=device)
        sample = generator.forward(z)
        plot_batch(sample)


## Your solution

In the cell below, implement your solution. Make sure that the `fit` method prepares the image colorization model, while the `predict` method uses the model to colorize the input data `x_gray`.


In [ ]:
class YourModel(torch.nn.Module):
    def __init__(self, generator):
        super().__init__()
        self.generator = generator

    def fit(self):
        # Fit your image colorization method here
        pass

    def predict(self, x_gray):
        # Use your method here to colorize the images x_gray
        return x_gray.repeat(1, 3, 1, 1) # by default returns the input batch as RGB

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

your_model = YourModel(generator)
your_model.fit()

if not FINAL_EVALUATION_MODE:
    benchmark_solution(your_model, device)